# 01. Raw Data Audit — Herbal Supplements

This notebook audits the two category-filtered Amazon source tables before any transformation. The parent item (`parent_asin`) is the downstream retrieval and evaluation unit, while the review table links users, parent items, timestamps, and review content. No feature construction, target selection, retrieval, or model fitting occurs here.

The audit covers schemas and data types, missingness, cardinality, text-field lengths, structured metadata fields, and the distributions of reviews across parent items and users. Ratings, verification status, helpful votes, and timestamps are inspected only to characterize the raw inputs; they are not authorized as downstream recommendation evidence.

The raw item table contains `store` rather than a direct `brand` column. The optional direct-brand check therefore produces no table in this run. Notebook 02 constructs the primary Brand field from `store` and mapped `details` metadata.

The stored execution contains 27,407 item records and 593,363 review interactions spanning 1 January 2019 to 31 December 2022. The notebook exports descriptive audit summaries and diagnostics without modifying the source tables.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# ==== Load Libraries ====
from pathlib import Path
from collections import Counter
import ast
import json
import platform
import socket

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

print("Libraries loaded.")

Libraries loaded.


In [4]:
# ==== Define Inputs and Output Paths ====
NOTEBOOK_NAME = "01_data_check_herbal.ipynb"
CATEGORY_ID = "herbal"
CATEGORY_KEY = "herbal_supplements"
CATEGORY_FOLDER = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"
STAGE = "stage0_data_check"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
SUMMARY_DIR = PROJECT_ROOT / "outputs" / "stage0_data_check"

ITEMS_PATH = DATA_DIR / "items_Herbal_Supplements_W2_2019_2022.parquet"
REVIEWS_PATH = DATA_DIR / "reviews_Herbal_Supplements_W2_2019_2022.parquet"
RESULTS_OVERALL_PATH = SUMMARY_DIR / "results_overall.csv"
DIAGNOSTICS_SUMMARY_PATH = SUMMARY_DIR / "diagnostics_summary.csv"

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "stage": STAGE,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "summary_dir": str(SUMMARY_DIR),
    "items_path": str(ITEMS_PATH),
    "reviews_path": str(REVIEWS_PATH),
}

REQUIRED_INPUT_PATHS = {
    "items_source": ITEMS_PATH,
    "reviews_source": REVIEWS_PATH,
}
missing_inputs = {name: str(path) for name, path in REQUIRED_INPUT_PATHS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ITEMS_PATH:", ITEMS_PATH)
print("REVIEWS_PATH:", REVIEWS_PATH)
print("SUMMARY_DIR:", SUMMARY_DIR)

PROJECT_ROOT: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements
ITEMS_PATH: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/items_Herbal_Supplements_W2_2019_2022.parquet
REVIEWS_PATH: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/reviews_Herbal_Supplements_W2_2019_2022.parquet
SUMMARY_DIR: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_data_check


In [5]:
# ==== Load Raw Tables ====
items = pd.read_parquet(ITEMS_PATH)
reviews = pd.read_parquet(REVIEWS_PATH)

print("items shape :", items.shape)
print("reviews shape:", reviews.shape)

items shape : (27407, 12)
reviews shape: (593363, 9)


In [6]:
# ==== Inspect Columns and Data Types ====
print("Items columns:")
print(items.columns.tolist())
print()

print("Reviews columns:")
print(reviews.columns.tolist())
print()

print("Items dtypes:")
print(items.dtypes)
print()

print("Reviews dtypes:")
print(reviews.dtypes)

Items columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

Reviews columns:
['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Items dtypes:
main_category      object
title              object
average_rating     object
rating_number      object
features           object
description        object
price              object
store              object
categories         object
details            object
parent_asin        object
bought_together    object
dtype: object

Reviews dtypes:
rating               object
title                object
text                 object
asin                 object
parent_asin          object
user_id              object
timestamp            object
helpful_vote         object
verified_purchase    object
dtype: object


In [7]:
# ==== Preview Source Records ====
print("Items head:")
display(items.head(5))

print("Reviews head:")
display(reviews.head(5))

Items head:


,main_category,title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin,bought_together
0,Health & Personal Care,Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack),4.6,375,"[""Value Pack 2 Bottles of Bio-Curcumin Elite 400 mg, 60 Vegetarian Capsules (2x60)"", ""Up to 7 times more absorbable than conventional Curcumin supplements"", ""During the summer months products may arrive warm but Amazon stores and ships products in accordance with manufacturers' recommendations, ...","[""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produces more bioavailability and tissue distribution of free curcuminoids than unformulated curcumin, and they last much longer in the bloodstream. That translat...",46.98,Life Extension,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Curcumin""]","{""Item Form"": ""capsules"", ""Brand"": ""Life Extension"", ""Age Range (Description)"": ""Adult"", ""Material Feature"": ""Vegetarian"", ""Recommended Uses For Product"": ""Healthy Inflammatory"", ""Is Discontinued By Manufacturer"": ""No"", ""Product Dimensions"": ""2 x 4 x 2 inches; 5.1 Ounces"", ""Item model number"": ""...",B01LYS06EF,None
1,Health & Personal Care,"Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces…",3.9,95,"[""The flavor is somewhat peppery and slightly sweet, with a strong and spicy aroma."", ""Apart from being a great flavoring agent, ginger root also makes a great daily supplement."", ""The Ginger Root Extract Liquid is made of Ginger root liquid extract, purified Water and Propylene glycol."", ""The s...",[],None,Naturevibe Botanicals,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Ginger""]","{""Item Form"": ""Liquid"", ""Brand"": ""Naturevibe Botanicals"", ""Age Range (Description)"": ""Adult"", ""Material Feature"": ""Liquid"", ""Number of Items"": ""1"", ""Package Dimensions"": ""5 x 3.78 x 2.32 inches; 2 Ounces"", ""Date First Available"": ""January 15, 2021"", ""Manufacturer"": ""Naturevibe Botanicals""}",B08T67YDQF,None
2,Health & Personal Care,"Nature's Way Artichoke, 60 Capsules (Pack of 2)",4.2,43,[],[],None,Nature's Way,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Artichoke""]","{""Item Form"": ""Capsule"", ""Brand"": ""Nature's Way"", ""Age Range (Description)"": ""Adult"", ""Recommended Uses For Product"": ""Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.*"", ""Number of Items"": ""2"", ""Is Discontinued By Manufacturer"": ""No"", ""Product Dimensions"": ""...",B002LIMQQA,None
3,Health & Personal Care,"Mushroom Revival, Organic, Calm Reishi Tincture, 2 Fluid Ounces, Non-GMO, Vegan, Kosher, Gluten Free, Keto, Dairy Free",3.8,80,"[""RELAX with the power of Reishi. Feel like a zen monk smiling throughout your day. Deflect occassional stress with your own personal force field. Melt away tension, and come back to feeling your true self."", ""Easiest thing you can do all day"", ""Organic, non-gmo, vegan, kosher, gluten free, keto...",[],34.95,Mushroom Revival,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Mushrooms""]","{""Item Form"": ""Drop"", ""Brand"": ""Mushroom Revival"", ""Age Range (Description)"": ""Adult"", ""Number of Items"": ""1"", ""Flavor"": ""Reishi Calm"", ""Package Dimensions"": ""4.9 x 2.7 x 2.5 inches; 3.99 Ounces"", ""Date First Available"": ""April 23, 2019"", ""Manufacturer"": ""Mushroom Revival""}",B0B1236V1Q,None
4,Health & Personal Care,"RYUKAKUSAN Herbal Drop, Yuzu, 11 Count",2.7,2,"[""Contains effective Chinese herbal great for cough, voice recovery, breath refreshing and smoker"", ""Well know Original Japanese herbal throat drops"", ""Manufactured in Japan""]",[],None,Ryukakusan,"[""Health & Household"", ""Vitamins, Minerals & Supplements"

Reviews head:


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,BLOCKS IRON ABSORPTION...BE AWARE!,"Although this WAS one of my favorite products for keeping the immune system up and reducing allergies, it blocks iron absorption. Caffeine and calcium also block iron absorption. If you take this, be sure your FERRITIN levels are very high...as some people become symptomatic (severe anxiety, rap...",B0019LWTQW,B07C1XKDBV,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1593364174638,176,True
1,5.0,I CRAVE this stuff!,Blends nicely (I use my frother stick) with No grit or residue. Tastes fresh. ( I have tried others that tasted so swampy that I couldn't drink them. This one does not. I actually look forward to drinking it nice and cold!,B01N33C6EP,B09W7GK6WL,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1654227757388,4,True
2,5.0,drops,Thought I would try these. I am surprised how little you use. Great product. I'll be back to get more if I ever use all of this.,B01IF9UBW4,B09GTZSDF2,AEKDF2ANHCJWJQSPSMPV5TNTNZJA,1595802315366,0,True
3,1.0,Nausea on stomach,Hard to take I take 3 morning/3 evening. Causes upset stomach even after eating food. What can I do? I used KappArest no problem,B073G716Z1,B098PR8VHG,AHTH75HJJ5QPIP3ZDF2XNIBE3CWA,1628601567737,0,True
4,2.0,I would not buy this again,"This sadly just didn’t work for me. I was previously taking Chlorophyll 100mg pills and felt they worked better, less messy, were more convenient, and cheaper. This only has 50mg chlorophyll per 15 drops, but I thought the liquid would be more potent. It is not. I’m disappointed.",B01AFI3IEG,B01AFI3IEG,AEYORY2AVPMCPDV57CE337YU5LXA,1577586399246,3,True


In [8]:
# ==== Summarize Missing Values ====
items_nulls = pd.DataFrame({
    "null_count": items.isna().sum(),
    "null_ratio": items.isna().mean().round(4)
}).sort_values(["null_ratio", "null_count"], ascending=False)

reviews_nulls = pd.DataFrame({
    "null_count": reviews.isna().sum(),
    "null_ratio": reviews.isna().mean().round(4)
}).sort_values(["null_ratio", "null_count"], ascending=False)

print("Items null summary:")
display(items_nulls)

print("Reviews null summary:")
display(reviews_nulls)

Items null summary:


,null_count,null_ratio
bought_together,27407,1.0000
price,12195,0.4450
main_category,1680,0.0613
store,98,0.0036
title,0,0.0000
average_rating,0,0.0000
rating_number,0,0.0000
features,0,0.0000
description,0,0.0000
categories,0,0.0000


Reviews null summary:


,null_count,null_ratio
rating,0,0.0
title,0,0.0
text,0,0.0
asin,0,0.0
parent_asin,0,0.0
user_id,0,0.0
timestamp,0,0.0
helpful_vote,0,0.0
verified_purchase,0,0.0


In [9]:
# ==== Summarize Field Cardinality ====
def safe_nunique(series):
    try:
        return series.nunique(dropna=True)
    except Exception:
        return None

items_cardinality = pd.DataFrame({
    "dtype": items.dtypes.astype(str),
    "nunique": [safe_nunique(items[c]) for c in items.columns]
}).sort_values("nunique", ascending=False)

reviews_cardinality = pd.DataFrame({
    "dtype": reviews.dtypes.astype(str),
    "nunique": [safe_nunique(reviews[c]) for c in reviews.columns]
}).sort_values("nunique", ascending=False)

print("Items cardinality:")
display(items_cardinality)

print("Reviews cardinality:")
display(reviews_cardinality)

Items cardinality:


,dtype,nunique
parent_asin,object,27407
title,object,26635
details,object,26433
features,object,20244
description,object,14059
store,object,6526
price,object,3214
rating_number,object,2152
categories,object,74
average_rating,object,40


Reviews cardinality:


,dtype,nunique
timestamp,object,568906
text,object,522220
user_id,object,481279
title,object,313968
asin,object,22847
parent_asin,object,19622
helpful_vote,object,501
rating,object,5
verified_purchase,object,2


In [10]:
# ==== Summarize Text-Field Lengths ====
def text_len_stats(df, col):
    s = df[col].fillna("").astype(str)
    return pd.Series({
        "non_null": df[col].notna().sum(),
        "avg_len": round(s.str.len().mean(), 2),
        "median_len": round(s.str.len().median(), 2),
        "max_len": s.str.len().max()
    })

length_stats = {}

for col in ["title", "features", "description", "details", "categories", "brand"]:
    if col in items.columns:
        length_stats[f"items::{col}"] = text_len_stats(items, col)

for col in ["title", "text"]:
    if col in reviews.columns:
        length_stats[f"reviews::{col}"] = text_len_stats(reviews, col)

length_stats_df = pd.DataFrame(length_stats).T
display(length_stats_df)

,non_null,avg_len,median_len,max_len
items::title,27407.0,107.91,99.0,683.0
items::features,27407.0,553.94,400.0,4834.0
items::description,27407.0,341.51,95.0,6490.0
items::details,27407.0,312.39,319.0,1957.0
items::categories,27407.0,92.83,93.0,120.0
reviews::title,593363.0,21.54,17.0,100.0
reviews::text,593363.0,178.28,105.0,12745.0


In [11]:
# ==== Inspect the Structured Details Field ====
def parse_maybe_dict(x):
    if pd.isna(x):
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            return ast.literal_eval(x)
        except Exception:
            try:
                return json.loads(x)
            except Exception:
                return None
    return None

details_parsed = items["details"].apply(parse_maybe_dict) if "details" in items.columns else pd.Series(dtype="object")

print("Parsed details non-null count:", details_parsed.notna().sum())

sample_details = details_parsed.dropna().head(10).tolist()
for i, d in enumerate(sample_details, start=1):
    print(f"\nSample details {i}:")
    print(type(d))
    print(list(d.items())[:10] if isinstance(d, dict) else d)

Parsed details non-null count: 27407

Sample details 1:
<class 'dict'>
[('Item Form', 'capsules'), ('Brand', 'Life Extension'), ('Age Range (Description)', 'Adult'), ('Material Feature', 'Vegetarian'), ('Recommended Uses For Product', 'Healthy Inflammatory'), ('Is Discontinued By Manufacturer', 'No'), ('Product Dimensions', '2 x 4 x 2 inches; 5.1 Ounces'), ('Item model number', '00407'), ('Date First Available', 'April 10, 2014'), ('Manufacturer', 'Life Extension')]

Sample details 2:
<class 'dict'>
[('Item Form', 'Liquid'), ('Brand', 'Naturevibe Botanicals'), ('Age Range (Description)', 'Adult'), ('Material Feature', 'Liquid'), ('Number of Items', '1'), ('Package Dimensions', '5 x 3.78 x 2.32 inches; 2 Ounces'), ('Date First Available', 'January 15, 2021'), ('Manufacturer', 'Naturevibe Botanicals')]

Sample details 3:
<class 'dict'>
[('Item Form', 'Capsule'), ('Brand', "Nature's Way"), ('Age Range (Description)', 'Adult'), ('Recommended Uses For Product', 'Artichoke extract is standar

In [12]:
# ==== Count Structured-Detail Keys ====
from collections import Counter

detail_key_counter = Counter()

for d in details_parsed.dropna():
    if isinstance(d, dict):
        detail_key_counter.update(d.keys())

detail_key_df = pd.DataFrame(
    detail_key_counter.most_common(100),
    columns=["detail_key", "count"]
)

print("Top detail keys:")
display(detail_key_df)

Top detail keys:


,detail_key,count
0,Manufacturer,25540
1,Date First Available,24163
2,Brand,21482
3,Item Form,20066
4,Age Range (Description),18256
...,...,...
95,Target Audience,5
96,Water Resistance Level,5
97,Whats in the box,5
98,Run time,5


In [13]:
# ==== Inspect List-Like Metadata Fields ====
for col in ["features", "description"]:
    if col in items.columns:
        print(f"\nColumn structure check: {col}")
        sample_vals = items[col].dropna().head(10).tolist()
        for i, v in enumerate(sample_vals, start=1):
            print(f"\nSample {i} type={type(v)}")
            print(str(v)[:500])


Column structure check: features

Sample 1 type=<class 'str'>
["Value Pack 2 Bottles of Bio-Curcumin Elite 400 mg, 60 Vegetarian Capsules (2x60)", "Up to 7 times more absorbable than conventional Curcumin supplements", "During the summer months products may arrive warm but Amazon stores and ships products in accordance with manufacturers' recommendations, when provided."]

Sample 2 type=<class 'str'>
["The flavor is somewhat peppery and slightly sweet, with a strong and spicy aroma.", "Apart from being a great flavoring agent, ginger root also makes a great daily supplement.", "The Ginger Root Extract Liquid is made of Ginger root liquid extract, purified Water and Propylene glycol.", "The suggested use of ginger root extract liquid is : As a dietary supplement take 1-2 ml, 2-3 times a day in small amount of water.", "Naturevibe Botanicals Ginger Root Extract Liquid comes in a bottle of 2oz 

Sample 3 type=<class 'str'>
[]

Sample 4 type=<class 'str'>
["RELAX with the power of Reishi.

In [14]:
# ==== Inspect Category and Co-Purchase Fields ====
for col in ["categories", "bought_together"]:
    if col in items.columns:
        print(f"\nColumn structure check: {col}")
        sample_vals = items[col].dropna().head(10).tolist()
        for i, v in enumerate(sample_vals, start=1):
            print(f"\nSample {i} type={type(v)}")
            print(str(v)[:500])


Column structure check: categories

Sample 1 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Curcumin"]

Sample 2 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Ginger"]

Sample 3 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Artichoke"]

Sample 4 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Mushrooms"]

Sample 5 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Alfalfa"]

Sample 6 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Milk Thistle"]

Sample 7 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplements", "Mushrooms"]

Sample 8 type=<class 'str'>
["Health & Household", "Vitamins, Minerals & Supplements", "Herbal Supplement

In [15]:
# ==== Summarize Reviews per Parent Item ====
if "parent_asin" in reviews.columns:
    review_per_item = reviews.groupby("parent_asin").size().rename("n_reviews").reset_index()

    print("Review count per parent_asin summary:")
    print(review_per_item["n_reviews"].describe())

    print("\nTop reviewed items:")
    display(review_per_item.sort_values("n_reviews", ascending=False).head(20))

    print("\nBottom reviewed items:")
    display(review_per_item.sort_values("n_reviews", ascending=True).head(20))

Review count per parent_asin summary:
count    19622.000000
mean        30.239680
std        209.018606
min          1.000000
25%          1.000000
50%          4.000000
75%         14.000000
max      11601.000000
Name: n_reviews, dtype: float64

Top reviewed items:


,parent_asin,n_reviews
9404,B078K93HFD,11601
19498,B0C769QRX5,9882
18429,B0B8JSPBRM,8693
10722,B07H38W4GK,7148
17075,B09K64N86H,6344
19499,B0C76LM2HW,6342
16025,B092RNPW6M,6128
11131,B07KX8MXCM,5739
16198,B094LDR8SH,5513
6401,B01AFI3IEG,5347



Bottom reviewed items:


,parent_asin,n_reviews
19608,B0CG23M717,1
19604,B0CFTQGNSJ,1
18,B00012NDGA,1
16,B00012NCM0,1
15,B00012NCK2,1
14,B00012NCCA,1
13,B00008MO0B,1
10,B0000533AJ,1
5,5230130660,1
3,3597126197,1


In [16]:
# ==== Summarize Reviews per User ====
if "user_id" in reviews.columns:
    review_per_user = reviews.groupby("user_id").size().rename("n_reviews").reset_index()

    print("Review count per user summary:")
    print(review_per_user["n_reviews"].describe())

    repeat_bucket = pd.cut(
        review_per_user["n_reviews"],
        bins=[0, 1, 4, 9, 19, review_per_user["n_reviews"].max()],
        labels=["1", "2-4", "5-9", "10-19", "20+"],
        include_lowest=True
    )

    user_bucket_summary = repeat_bucket.value_counts(dropna=False).sort_index().reset_index()
    user_bucket_summary.columns = ["repeat_bucket", "n_users"]
    user_bucket_summary["user_ratio"] = user_bucket_summary["n_users"] / user_bucket_summary["n_users"].sum()

    print("\nUser repeat bucket summary:")
    display(user_bucket_summary)

Review count per user summary:
count    481279.000000
mean          1.232888
std           1.991929
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         920.000000
Name: n_reviews, dtype: float64

User repeat bucket summary:


,repeat_bucket,n_users,user_ratio
0,1,416324,0.865037
1,2-4,61065,0.126881
2,5-9,3022,0.006279
3,10-19,624,0.001297
4,20+,244,0.000507


In [17]:
# ==== Audit Raw Review Fields and Time Coverage ====
if "rating" in reviews.columns:
    print("Rating distribution:")
    display(reviews["rating"].value_counts(dropna=False).sort_index())

if "verified_purchase" in reviews.columns:
    print("Verified purchase distribution:")
    display(reviews["verified_purchase"].value_counts(dropna=False))

if "helpful_vote" in reviews.columns:
    print("Helpful vote summary:")
    print(pd.to_numeric(reviews["helpful_vote"], errors="coerce").describe())

if "timestamp" in reviews.columns:
    ts = pd.to_datetime(pd.to_numeric(reviews["timestamp"], errors="coerce"), unit="ms", errors="coerce")
    print("Timestamp min:", ts.min())
    print("Timestamp max:", ts.max())
    print("Review year distribution:")
    display(ts.dt.year.value_counts().sort_index())

Rating distribution:


,count
rating,
1.0,51428
2.0,17264
3.0,26626
4.0,57306
5.0,440739


Verified purchase distribution:


,count
verified_purchase,
True,551183
False,42180


Helpful vote summary:
count    593363.000000
mean          2.071093
std          17.842430
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        4070.000000
Name: helpful_vote, dtype: float64
Timestamp min: 2019-01-01 00:06:37.122000
Timestamp max: 2022-12-31 23:56:10.358000
Review year distribution:


,count
timestamp,
2019,116265
2020,191812
2021,157537
2022,127749


In [18]:
# ==== Check for an Optional Direct Brand Field ====
if "brand" in items.columns:
    brand_counts = (
        items["brand"]
        .fillna("")
        .astype(str)
        .str.strip()
        .value_counts()
        .reset_index()
    )
    brand_counts.columns = ["brand", "n_items"]

    print("Top brands by item count:")
    display(brand_counts.head(30))

In [19]:
# ==== Export Audit Summaries ====
output_dir = SUMMARY_DIR
created_outputs = []

base_outputs = {
    "items_null_summary.csv": items_nulls,
    "reviews_null_summary.csv": reviews_nulls,
    "items_cardinality.csv": items_cardinality,
    "reviews_cardinality.csv": reviews_cardinality,
    "text_length_stats.csv": length_stats_df,
    "detail_key_frequency.csv": detail_key_df,
    "items_preview.csv": items.head(100),
    "reviews_preview.csv": reviews.head(100),
}

for filename, df in base_outputs.items():
    path = output_dir / filename
    df.to_csv(path, index=True if filename.endswith(("null_summary.csv", "cardinality.csv", "text_length_stats.csv")) else False, encoding="utf-8-sig")
    created_outputs.append(path)

optional_outputs = {
    "review_per_item.csv": globals().get("review_per_item"),
    "review_per_user.csv": globals().get("review_per_user"),
    "user_repeat_bucket_summary.csv": globals().get("user_bucket_summary"),
    "brand_item_counts.csv": globals().get("brand_counts"),
}

for filename, df in optional_outputs.items():
    if df is not None:
        path = output_dir / filename
        df.to_csv(path, index=False, encoding="utf-8-sig")
        created_outputs.append(path)

results_overall = pd.DataFrame([{
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "stage": STAGE,
    "input_items_path": str(ITEMS_PATH),
    "input_reviews_path": str(REVIEWS_PATH),
    "output_root": str(output_dir),
    "n_rows_items": len(items),
    "n_rows_reviews": len(reviews),
    "n_cols_items": items.shape[1],
    "n_cols_reviews": reviews.shape[1],
    "status": "completed",
}])
results_overall.to_csv(RESULTS_OVERALL_PATH, index=False, encoding="utf-8-sig")
created_outputs.append(RESULTS_OVERALL_PATH)

diagnostics_summary = pd.DataFrame([
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "items_input_exists",
        "path": str(ITEMS_PATH),
        "check_passed": ITEMS_PATH.exists(),
        "row_count": len(items),
        "note": "curated items input",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "reviews_input_exists",
        "path": str(REVIEWS_PATH),
        "check_passed": REVIEWS_PATH.exists(),
        "row_count": len(reviews),
        "note": "curated reviews input",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "items_has_rows",
        "path": str(ITEMS_PATH),
        "check_passed": len(items) > 0,
        "row_count": len(items),
        "note": "basic row-count check",
    },
    {
        "notebook_name": NOTEBOOK_NAME,
        "category_id": CATEGORY_ID,
        "category_key": CATEGORY_KEY,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "stage": STAGE,
        "check_name": "reviews_has_rows",
        "path": str(REVIEWS_PATH),
        "check_passed": len(reviews) > 0,
        "row_count": len(reviews),
        "note": "basic row-count check",
    },
])
diagnostics_summary.to_csv(DIAGNOSTICS_SUMMARY_PATH, index=False, encoding="utf-8-sig")
created_outputs.append(DIAGNOSTICS_SUMMARY_PATH)



print("Data check notebook completed.")
print("Items rows:", len(items))
print("Reviews rows:", len(reviews))
print("Summary directory:", SUMMARY_DIR)
print("Results summary:", RESULTS_OVERALL_PATH)
print("Diagnostics summary:", DIAGNOSTICS_SUMMARY_PATH)

display(results_overall)
display(diagnostics_summary)

Data check notebook completed.
Items rows: 27407
Reviews rows: 593363
Summary directory: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_data_check
Results summary: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_data_check/results_overall.csv
Diagnostics summary: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_data_check/diagnostics_summary.csv


,notebook_name,category_id,category_key,category_folder,category_label,stage,input_items_path,input_reviews_path,output_root,n_rows_items,n_rows_reviews,n_cols_items,n_cols_reviews,status
0,01_data_check_herbal.ipynb,herbal,herbal_supplements,herbal_supplements,Herbal Supplements,stage0_data_check,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/items_Herbal_Supplements_W2_2019_2022.parquet,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/reviews_Herbal_Supplements_W2_2019_2022.parquet,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_data_check,27407,593363,12,9,completed


,notebook_name,category_id,category_key,category_folder,category_label,stage,check_name,path,check_passed,row_count,note
0,01_data_check_herbal.ipynb,herbal,herbal_supplements,herbal_supplements,Herbal Supplements,stage0_data_check,items_input_exists,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/items_Herbal_Supplements_W2_2019_2022.parquet,True,27407,curated items input
1,01_data_check_herbal.ipynb,herbal,herbal_supplements,herbal_supplements,Herbal Supplements,stage0_data_check,reviews_input_exists,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/reviews_Herbal_Supplements_W2_2019_2022.parquet,True,593363,curated reviews input
2,01_data_check_herbal.ipynb,herbal,herbal_supplements,herbal_supplements,Herbal Supplements,stage0_data_check,items_has_rows,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/items_Herbal_Supplements_W2_2019_2022.parquet,True,27407,basic row-count check
3,01_data_check_herbal.ipynb,herbal,herbal_supplements,herbal_supplements,Herbal Supplements,stage0_data_check,reviews_has_rows,/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/reviews_Herbal_Supplements_W2_2019_2022.parquet,True,593363,basic row-count check
